# BioLiteNet — Biomedical Waste Classification
**MobileNetV3-Large + CBAM + Hazard-Aware Focal Loss (HAFL)**

This notebook contains the complete pipeline:
1. Configuration
2. Dataset extraction & splitting
3. Model definition (CBAM + BioLiteNet)
4. Loss functions (HAFL & Focal Loss)
5. Trainer (train / evaluate / test)
6. Run training (ablation variants)
7. Results table generator


## 0. Install Dependencies

In [17]:
# Run this cell only if packages are not yet installed
# !pip install torch torchvision scikit-learn pillow numpy


## 1. Configuration

In [18]:
import os

# dataset paths
DATASET_ZIP = r"e:\- CS\6th Semester\ML Lab\Bio Medical Waste - Dataset\archive.zip"
DATASET_DIR = r"e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\data"
OUTPUT_DIR  = r"e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\outputs"

# image size for MobileNetV3
IMG_SIZE = 224
USE_BACKGROUND_REMOVED = True

# training settings
BATCH_SIZE   = 16
NUM_EPOCHS   = 30
LR           = 1e-3
LR_MIN       = 1e-6
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.4
NUM_WORKERS  = 0   # must be 0 on Windows
SEED         = 42
LOG_INTERVAL = 10

# dataset split
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# focal loss gamma parameter
FOCAL_GAMMA = 2.0

# hazard weights (h_c) for each class based on WHO biomedical waste guidelines
# higher value = model penalized more for missing that class
HAZARD_WEIGHTS = {
    # sharps - highest risk (direct injury + bloodborne pathogens)
    "used_syringes":           3.0,
    "scalpels":                3.0,
    "episiotomy_scissors":     3.0,
    "mayo_scissors":           3.0,
    "stitch_removal_scissors": 3.0,
    "tweezers":                2.8,
    "forceps":                 2.8,
    "hemostats":               2.8,

    # pathological - infection risk
    "human_organs":            2.5,
    "blood_soaked_bandages":   2.5,

    # chemical - environmental hazard
    "disinfectant_bottles":    2.0,
    "ampoules_full":           2.0,
    "ampuoles_broken":         2.2,

    # pharmaceutical - ingestion risk
    "expired_tablets":         1.5,
    "vaccine_or_medicine_vials": 1.5,
    "syrup_bottles":           1.5,
    "iv_bottles":              1.5,

    # general waste - lower risk
    "used_masks":              1.2,
    "used_medical_gloves":     1.2,
    "uncategorized_or_overlapping": 1.3,
    "used_medical_paper":      1.0,
    "general_organic_waste":   1.0,
    "waterbottles":            1.0,
}

# CBAM is inserted after these stages of MobileNetV3-Large
CBAM_STAGES    = [3, 6, 12]
CBAM_REDUCTION = 16

# flags to control which variant to train
USE_CBAM = True
USE_HAFL = True


## 2. Dataset — Extraction & Splitting

In [19]:
import os, zipfile, shutil, random, json
# config variables imported from Cell 1 (already defined above)

random.seed(SEED)


def extract_dataset():
    if os.path.exists(DATASET_DIR) and os.listdir(DATASET_DIR):
        print("[INFO] Dataset already extracted at:", DATASET_DIR)
        return
    print("[INFO] Extracting dataset...")
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        zf.extractall(DATASET_DIR)
    print("[INFO] Done.")


def get_class_root():
    subfolder = "Baackground_Removed" if USE_BACKGROUND_REMOVED else "Raw"
    root = os.path.join(DATASET_DIR, "Dataset", subfolder)
    if not os.path.isdir(root):
        parent = os.path.join(DATASET_DIR, "Dataset")
        for d in os.listdir(parent):
            if d.lower().replace(" ", "") == subfolder.lower().replace(" ", ""):
                root = os.path.join(parent, d)
                break
    return root


def build_splits():
    root    = get_class_root()
    classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    print(f"[INFO] Found {len(classes)} classes")

    split_root = os.path.join(OUTPUT_DIR, "splits")
    if os.path.exists(split_root) and os.listdir(split_root):
        print("[INFO] Splits already exist at:", split_root)
        return split_root, classes

    stats = {}
    for cls in classes:
        cls_dir = os.path.join(root, cls)
        images  = [f for f in os.listdir(cls_dir)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
        random.shuffle(images)

        n       = len(images)
        n_train = int(n * TRAIN_RATIO)
        n_val   = int(n * VAL_RATIO)
        splits  = {
            "train": images[:n_train],
            "val":   images[n_train: n_train + n_val],
            "test":  images[n_train + n_val:]
        }
        stats[cls] = {s: len(imgs) for s, imgs in splits.items()}

        for split, imgs in splits.items():
            dst_dir = os.path.join(split_root, split, cls)
            os.makedirs(dst_dir, exist_ok=True)
            for img in imgs:
                src = os.path.join(cls_dir, img)
                dst = os.path.join(dst_dir, img)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(os.path.join(OUTPUT_DIR, "split_stats.json"), 'w') as f:
        json.dump(stats, f, indent=2)

    print(f"\n{'Class':<40} {'Train':>6} {'Val':>6} {'Test':>6} {'h_c':>5}")
    print("-" * 65)
    for cls in classes:
        hc = HAZARD_WEIGHTS.get(cls, 1.0)
        s  = stats[cls]
        print(f"{cls:<40} {s['train']:>6} {s['val']:>6} {s['test']:>6} {hc:>5.1f}")

    return split_root, classes


if __name__ == "__main__":
    extract_dataset()
    split_root, classes = build_splits()
    print("\nDataset ready at:", split_root)
    print("Total classes:", len(classes))


[INFO] Dataset already extracted at: e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\data
[INFO] Found 23 classes
[INFO] Splits already exist at: e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\outputs\splits

Dataset ready at: e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\outputs\splits
Total classes: 23


## 3. Model — CBAM + BioLiteNet

In [20]:
import torch
import torch.nn as nn
import torchvision.models as models


class ChannelGate(nn.Module):
    # channel attention: tells the model which feature channels are important
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        mid = max(in_channels // reduction, 4)
        self.shared_mlp = nn.Sequential(
            nn.Linear(in_channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_channels, bias=False),
        )

    def forward(self, x):
        B, C, H, W = x.shape
        avg_out = self.shared_mlp(x.mean(dim=[2, 3]))
        max_out = self.shared_mlp(x.amax(dim=[2, 3]))
        scale = torch.sigmoid(avg_out + max_out).view(B, C, 1, 1)
        return x * scale


class SpatialGate(nn.Module):
    # spatial attention: tells the model where to focus in the image
    def __init__(self, kernel_size=7):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=pad, bias=False)
        self.bn = nn.BatchNorm2d(1)

    def forward(self, x):
        avg_map = x.mean(dim=1, keepdim=True)
        max_map = x.amax(dim=1, keepdim=True)
        spatial = torch.cat([avg_map, max_map], dim=1)
        scale = torch.sigmoid(self.bn(self.conv(spatial)))
        return x * scale


class CBAM(nn.Module):
    # applies channel attention then spatial attention
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.channel = ChannelGate(in_channels, reduction)
        self.spatial = SpatialGate()

    def forward(self, x):
        x = self.channel(x)
        x = self.spatial(x)
        return x


class BioLiteNet(nn.Module):
    def __init__(self, num_classes=23, dropout_rate=0.4, use_cbam=True,
                 cbam_stages=None, cbam_reduction=16, pretrained=True):
        super().__init__()

        if cbam_stages is None:
            cbam_stages = [3, 6, 12]

        self.use_cbam = use_cbam
        self.cbam_stages = set(cbam_stages)

        # load pretrained MobileNetV3-Large as backbone
        weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = models.mobilenet_v3_large(weights=weights)

        # split backbone into individual stages so we can insert CBAM between them
        self.stages = nn.ModuleList()
        self.cbams  = nn.ModuleDict()

        for i, stage in enumerate(backbone.features.children()):
            self.stages.append(stage)
            if use_cbam and i in self.cbam_stages:
                out_ch = self._get_out_channels(stage)
                if out_ch is not None:
                    self.cbams[str(i)] = CBAM(out_ch, reduction=cbam_reduction)

        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.dropout    = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(960, num_classes)

    @staticmethod
    def _get_out_channels(module):
        # find the last conv layer to get the output channel count
        last_conv = None
        for m in module.modules():
            if isinstance(m, nn.Conv2d):
                last_conv = m
        if last_conv is not None:
            return last_conv.out_channels
        for m in module.modules():
            if isinstance(m, nn.BatchNorm2d):
                return m.num_features
        return None

    def forward(self, x):
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if self.use_cbam and str(i) in self.cbams:
                x = self.cbams[str(i)](x)
        x = self.pool(x)
        x = x.flatten(1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

    def count_parameters(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        cbam_p    = sum(p.numel() for p in self.cbams.parameters())
        return {
            "total_M":     round(total / 1e6, 2),
            "trainable_M": round(trainable / 1e6, 2),
            "cbam_M":      round(cbam_p / 1e6, 3),
        }


## 4. Loss Functions — HAFL & Focal Loss

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class HazardAwareFocalLoss(nn.Module):
    # HAFL formula: L = -h_c * alpha_t * (1 - p_t)^gamma * log(p_t)
    # h_c is the hazard weight for each class (based on WHO guidelines)
    def __init__(self, hazard_weights, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma     = gamma
        self.reduction = reduction

        h = torch.tensor(hazard_weights, dtype=torch.float32)
        self.register_buffer("hazard_weights", h)

        if alpha is not None:
            a = torch.tensor(alpha, dtype=torch.float32)
        else:
            a = torch.ones(len(hazard_weights), dtype=torch.float32)
        self.register_buffer("alpha", a)

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        probs     = torch.exp(log_probs)

        p_t     = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        log_p_t = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)

        alpha_t = self.alpha[targets]
        h_c     = self.hazard_weights[targets]

        focal_weight = (1.0 - p_t) ** self.gamma
        loss = -h_c * alpha_t * focal_weight * log_p_t

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


def build_hafl(class_names, config_hazard_weights, gamma=2.0):
    # builds HAFL with h_c values matched to dataloader class order
    h_values = []
    print("\n[HAFL] Hazard weight assignment:")
    print(f"  {'Idx':<4} {'Class':<40} {'h_c':>5}")
    print("  " + "-" * 52)
    for idx, cls in enumerate(class_names):
        hc = config_hazard_weights.get(cls, 1.0)
        h_values.append(hc)
        print(f"  {idx:<4} {cls:<40} {hc:>5.1f}")
    return HazardAwareFocalLoss(hazard_weights=h_values, gamma=gamma)


class FocalLoss(nn.Module):
    # standard focal loss used in ablation variants without HAFL
    def __init__(self, num_classes, gamma=2.0):
        super().__init__()
        weights   = [1.0] * num_classes
        self.hafl = HazardAwareFocalLoss(weights, gamma=gamma)

    def forward(self, logits, targets):
        return self.hafl(logits, targets)


## 5. Trainer

In [22]:
import os, time, json, csv, random, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# config variables already defined in Cell 1
# BioLiteNet already defined in Cell 3
# build_hafl, FocalLoss already defined in Cell 4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# classes considered sharps - used to compute sharps-F1 separately
SHARP_CLASSES = {
    "used_syringes", "scalpels", "episiotomy_scissors",
    "mayo_scissors", "stitch_removal_scissors",
    "tweezers", "forceps", "hemostats"
}


def get_transforms():
    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.RandomRotation(degrees=15),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_tf, val_tf


def get_dataloaders(split_root):
    train_tf, val_tf = get_transforms()
    train_ds = datasets.ImageFolder(os.path.join(split_root, "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(split_root, "val"),   transform=val_tf)
    test_ds  = datasets.ImageFolder(os.path.join(split_root, "test"),  transform=val_tf)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_dl, val_dl, test_dl, train_ds.classes


def build_criterion(class_names, use_hafl, num_classes):
    if use_hafl:
        return build_hafl(class_names, HAZARD_WEIGHTS, gamma=FOCAL_GAMMA)
    print("[INFO] Using standard focal loss (no HAFL)")
    return FocalLoss(num_classes=num_classes, gamma=FOCAL_GAMMA)


def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch_idx, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

        if (batch_idx + 1) % LOG_INTERVAL == 0:
            print(f"  Epoch {epoch:02d} | Batch {batch_idx+1:>4}/{len(loader)} "
                  f"| Loss: {loss.item():.4f} | Acc: {correct/total*100:.1f}%")

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device, class_names):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            preds  = logits.argmax(dim=1)

            total_loss += loss.item() * imgs.size(0)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss   = total_loss / total
    accuracy   = correct / total
    macro_f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    per_cls_f1 = f1_score(all_labels, all_preds, average=None,    zero_division=0)
    f1_dict    = {cls: round(float(per_cls_f1[i]), 4) for i, cls in enumerate(class_names)}

    return avg_loss, accuracy, macro_f1, f1_dict, all_preds, all_labels


def sharps_f1(f1_dict):
    vals = [v for k, v in f1_dict.items() if k in SHARP_CLASSES]
    return round(float(np.mean(vals)), 4) if vals else 0.0


def train(split_root, variant_name="BioLiteNet",
          use_cbam=USE_CBAM, use_hafl=USE_HAFL, resume=True):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*55}")
    print(f"  Variant: {variant_name} | Device: {device}")
    print(f"  CBAM: {use_cbam} | HAFL: {use_hafl}")
    print(f"{'='*55}\n")

    train_dl, val_dl, test_dl, class_names = get_dataloaders(split_root)
    num_classes = len(class_names)
    print(f"[INFO] {num_classes} classes | Train: {len(train_dl)} batches "
          f"| Val: {len(val_dl)} | Test: {len(test_dl)}")

    model = BioLiteNet(
        num_classes=num_classes, dropout_rate=DROPOUT_RATE,
        use_cbam=use_cbam, cbam_stages=CBAM_STAGES,
        cbam_reduction=CBAM_REDUCTION, pretrained=True,
    ).to(device)
    print("[MODEL]", model.count_parameters())

    criterion = build_criterion(class_names, use_hafl, num_classes).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=LR_MIN)

    variant_dir = os.path.join(OUTPUT_DIR, variant_name)
    ckpt_dir    = os.path.join(variant_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    best_ckpt   = os.path.join(ckpt_dir, "best_model.pth")
    last_ckpt   = os.path.join(ckpt_dir, "last_epoch.pth")
    start_epoch = 1
    best_val_f1 = 0.0
    best_epoch  = 0

    # resume from last saved checkpoint if available
    resume_file = last_ckpt if os.path.exists(last_ckpt) else (
                  best_ckpt if os.path.exists(best_ckpt) else None)

    if resume and resume_file:
        print(f"[RESUME] Loading: {resume_file}")
        ckpt = torch.load(resume_file, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        try:
            optimizer.load_state_dict(ckpt["optimizer"])
        except Exception:
            print("[WARN] Optimizer state mismatch, starting optimizer fresh")
        start_epoch = ckpt["epoch"] + 1
        if os.path.exists(best_ckpt):
            best_ckpt_data = torch.load(best_ckpt, map_location="cpu", weights_only=False)
            best_val_f1    = best_ckpt_data["val_f1"]
            best_epoch     = best_ckpt_data["epoch"]
        else:
            best_val_f1 = ckpt["val_f1"]
            best_epoch  = ckpt["epoch"]
        for _ in range(ckpt["epoch"]):
            scheduler.step()
        print(f"[RESUME] Continuing from epoch {start_epoch} | Best F1 so far: {best_val_f1:.4f}")
    else:
        print("[INFO] Starting fresh training")

    if start_epoch > NUM_EPOCHS:
        print(f"[DONE] All {NUM_EPOCHS} epochs already complete.")
    else:
        csv_path   = os.path.join(variant_dir, "training_log.csv")
        write_mode = "a" if (resume and os.path.exists(csv_path) and
                             os.path.getsize(csv_path) > 0) else "w"
        csv_file   = open(csv_path, write_mode, newline="")
        csv_writer = csv.writer(csv_file)
        if write_mode == "w":
            csv_writer.writerow(["epoch", "train_loss", "train_acc", "val_loss",
                                  "val_acc", "val_macro_f1", "sharps_f1", "lr"])

        try:
            for epoch in range(start_epoch, NUM_EPOCHS + 1):
                t0 = time.time()
                tr_loss, tr_acc = train_one_epoch(model, train_dl, criterion, optimizer, device, epoch)
                vl_loss, vl_acc, vl_f1, f1_dict, _, _ = evaluate(model, val_dl, criterion, device, class_names)
                sf1 = sharps_f1(f1_dict)
                lr  = optimizer.param_groups[0]["lr"]
                scheduler.step()

                print(f"\nEpoch {epoch:02d}/{NUM_EPOCHS} [{time.time()-t0:.0f}s] "
                      f"TrainLoss={tr_loss:.4f} TrainAcc={tr_acc*100:.1f}% | "
                      f"ValAcc={vl_acc*100:.1f}% MacroF1={vl_f1:.4f} SharpsF1={sf1:.4f}")

                csv_writer.writerow([epoch, round(tr_loss, 6), round(tr_acc, 6),
                                     round(vl_loss, 6), round(vl_acc, 6),
                                     round(vl_f1, 6), round(sf1, 6), round(lr, 8)])
                csv_file.flush()

                # save checkpoint every epoch for resume support
                torch.save({
                    "epoch": epoch, "model_state": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "val_f1": vl_f1, "sharps_f1": sf1,
                    "class_names": class_names,
                    "config": {"use_cbam": use_cbam, "use_hafl": use_hafl,
                               "num_classes": num_classes},
                }, last_ckpt)

                if vl_f1 > best_val_f1:
                    best_val_f1 = vl_f1
                    best_epoch  = epoch
                    shutil.copy2(last_ckpt, best_ckpt)
                    print(f"  [BEST] Saved (MacroF1={vl_f1:.4f})")

        except KeyboardInterrupt:
            print("\n[STOPPED] Training interrupted. Progress saved.")
        finally:
            csv_file.close()

    if not os.path.exists(best_ckpt):
        print("[WARN] No best checkpoint found.")
        return None

    print(f"\n{'='*55}")
    print(f"  Test Evaluation (best epoch {best_epoch})")
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])

    te_loss, te_acc, te_f1, te_f1_dict, te_preds, te_labels = evaluate(
        model, test_dl, criterion, device, class_names)
    te_sf1 = sharps_f1(te_f1_dict)

    print(f"  Accuracy : {te_acc*100:.2f}%")
    print(f"  Macro-F1 : {te_f1:.4f}")
    print(f"  Sharps-F1: {te_sf1:.4f}")

    print("\n[Per-Class F1]")
    for cls, f1 in sorted(te_f1_dict.items(), key=lambda x: -x[1]):
        tag = " <- SHARP" if cls in SHARP_CLASSES else ""
        print(f"  {cls:<42}: {f1:.4f}{tag}")

    report = classification_report(te_labels, te_preds, target_names=class_names, zero_division=0)
    with open(os.path.join(variant_dir, "test_report.txt"), "w") as f:
        f.write(f"Variant: {variant_name}\n")
        f.write(f"Accuracy : {te_acc*100:.2f}%\n")
        f.write(f"Macro-F1 : {te_f1:.4f}\n")
        f.write(f"SharpsF1 : {te_sf1:.4f}\n\n")
        f.write(report)

    with open(os.path.join(variant_dir, "per_class_f1.json"), "w") as f:
        json.dump({"variant": variant_name, "macro_f1": te_f1,
                   "accuracy": te_acc, "sharps_f1": te_sf1,
                   "per_class": te_f1_dict}, f, indent=2)

    np.save(os.path.join(variant_dir, "confusion_matrix.npy"),
            confusion_matrix(te_labels, te_preds))

    print(f"\n[DONE] Outputs saved to: {variant_dir}")
    print(f"{'='*55}\n")

    return {"variant": variant_name, "accuracy": te_acc,
            "macro_f1": te_f1, "sharps_f1": te_sf1, "per_class_f1": te_f1_dict}


## 6. Run Training
Edit `variant` to one of: `'A'` (baseline), `'B'` (CBAM only), `'C'` (HAFL only), `'D'` (BioLiteNet full).
Set `run_all=True` to train all four variants for the ablation study.

In [23]:
ABLATION_VARIANTS = {
    "A": dict(name="A_baseline",   use_cbam=False, use_hafl=False),
    "B": dict(name="B_cbam_only",  use_cbam=True,  use_hafl=False),
    "C": dict(name="C_hafl_only",  use_cbam=False, use_hafl=True),
    "D": dict(name="D_BioLiteNet", use_cbam=True,  use_hafl=True),
}

# ── Choose what to run ──────────────────────────────
variant     = "D"    # single variant key: A / B / C / D
run_all     = False  # set True to train all 4 variants
skip_extract = False # set True if dataset already extracted

# ── Setup ──────────────────────────────────────────
if not skip_extract:
    extract_dataset()
split_root, classes = build_splits()
print(f"\n[INFO] Dataset ready. Classes: {len(classes)}")

# ── Train ──────────────────────────────────────────
variants_to_run = list(ABLATION_VARIANTS.keys()) if run_all else [variant]

results = {}
for key in variants_to_run:
    cfg = ABLATION_VARIANTS[key]
    print(f"\n{'#'*50}")
    print(f"  VARIANT {key}: {cfg['name']}")
    print(f"{'#'*50}")
    result = train(
        split_root   = split_root,
        variant_name = cfg["name"],
        use_cbam     = cfg["use_cbam"],
        use_hafl     = cfg["use_hafl"],
    )
    if result:
        results[cfg["name"]] = result

if results:
    print(f"\n{'='*55}")
    print("  SUMMARY")
    print(f"{'='*55}")
    for name, res in results.items():
        print(f"  {name:<25} Acc={res['accuracy']*100:.2f}%  "
              f"F1={res['macro_f1']:.4f}  SharpsF1={res['sharps_f1']:.4f}")

print(f"\n[DONE] All outputs in: {OUTPUT_DIR}")


[INFO] Dataset already extracted at: e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\data
[INFO] Found 23 classes
[INFO] Splits already exist at: e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\outputs\splits

[INFO] Dataset ready. Classes: 23

##################################################
  VARIANT D: D_BioLiteNet
##################################################

  Variant: D_BioLiteNet | Device: cpu
  CBAM: True | HAFL: True

[INFO] 23 classes | Train: 683 batches | Val: 147 | Test: 147
[MODEL] {'total_M': 3.0, 'trainable_M': 3.0, 'cbam_M': 0.002}

[HAFL] Hazard weight assignment:
  Idx  Class                                      h_c
  ----------------------------------------------------
  0    ampoules_full                              2.0
  1    ampuoles_broken                            2.2
  2    blood_soaked_bandages                      2.5
  3    disinfectant_bottles                       2.0
  4    episiotomy_scissors                        3.0
  5    expired_tablets    

c:\Users\hp\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[RESUME] Continuing from epoch 31 | Best F1 so far: 0.9687
[DONE] All 30 epochs already complete.

  Test Evaluation (best epoch 26)
  Accuracy : 96.97%
  Macro-F1 : 0.9680
  Sharps-F1: 0.9864

[Per-Class F1]
  stitch_removal_scissors                   : 1.0000 <- SHARP
  used_masks                                : 1.0000
  uncategorized_or_overlapping              : 0.9978
  episiotomy_scissors                       : 0.9948 <- SHARP
  hemostats                                 : 0.9948 <- SHARP
  iv_bottles                                : 0.9948
  scalpels                                  : 0.9948 <- SHARP
  used_syringes                             : 0.9897 <- SHARP
  forceps                                   : 0.9843 <- SHARP
  mayo_scissors                             : 0.9843 <- SHARP
  used_medical_gloves                       : 0.9841
  general_organic_waste                     : 0.9789
  disinfectant_bottles                      : 0.9738
  syrup_bottles                        

## 7. Results Table (run after training all variants)

In [24]:
import os, json
import numpy as np
import torch, time
from torchvision import transforms
from PIL import Image
# OUTPUT_DIR, CBAM_STAGES, CBAM_REDUCTION, DROPOUT_RATE already defined in Cell 1

VARIANTS = ["A_baseline", "B_cbam_only", "C_hafl_only", "D_BioLiteNet"]

def load_results():
    results = {}
    for v in VARIANTS:
        path = os.path.join(OUTPUT_DIR, v, "per_class_f1.json")
        if os.path.exists(path):
            with open(path) as f:
                results[v] = json.load(f)
        else:
            print(f"[WARN] Missing results for variant {v}: {path}")
    return results

def measure_inference_speed(variant_dir: str, num_classes: int):
    """Measure avg inference time on CPU (paper's efficiency claim)."""
    try:
        # BioLiteNet already defined in Cell 3
        use_cbam = "cbam" in variant_dir.lower() or "biolite" in variant_dir.lower()
        model = BioLiteNet(num_classes=num_classes, use_cbam=use_cbam,
                           cbam_stages=CBAM_STAGES, pretrained=False)
        ckpt_path = os.path.join(variant_dir, "checkpoints", "best_model.pth")
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
            model.load_state_dict(ckpt["model_state"])
        model.eval()
        dummy = torch.randn(1, 3, 224, 224)
        # Warmup
        with torch.no_grad():
            for _ in range(5):
                model(dummy)
        # Time 50 passes
        times = []
        with torch.no_grad():
            for _ in range(50):
                t0 = time.perf_counter()
                model(dummy)
                times.append(time.perf_counter() - t0)
        return round(np.mean(times) * 1000, 2)   # ms per image
    except Exception as e:
        return f"N/A ({e})"

def print_table1(results: dict):
    """Table 1: Main comparison table for the paper."""
    print("\n" + "=" * 100)
    print("TABLE 1 — BioLiteNet Ablation Study & Performance Comparison")
    print("=" * 100)
    header = f"{'Variant':<20} {'CBAM':^6} {'HAFL':^6} {'Accuracy':^10} {'Macro-F1':^10} {'Sharps-F1':^11} {'Inf. (ms)':^11}"
    print(header)
    print("-" * 100)

    rows = []
    for v in VARIANTS:
        if v not in results:
            continue
        r        = results[v]
        use_cbam = "cbam" in v.lower() or "biolite" in v.lower()
        use_hafl = "hafl" in v.lower() or "biolite" in v.lower()
        spd      = measure_inference_speed(os.path.join(OUTPUT_DIR, v), len(r["per_class"]))
        acc_str  = f"{r['accuracy']*100:.2f}%"
        f1_str   = f"{r['macro_f1']:.4f}"
        sf1_str  = f"{r['sharps_f1']:.4f}"
        spd_str  = f"{spd} ms" if isinstance(spd, float) else spd
        label    = v.replace("_", " ")
        cbam_s   = "✓" if use_cbam else "✗"
        hafl_s   = "✓" if use_hafl else "✗"
        row_str  = f"{label:<20} {cbam_s:^6} {hafl_s:^6} {acc_str:^10} {f1_str:^10} {sf1_str:^11} {spd_str:^11}"
        print(row_str)
        rows.append([label, cbam_s, hafl_s, acc_str, f1_str, sf1_str, spd_str])

    print("=" * 100)

    # Save as CSV
    import csv
    csv_path = os.path.join(OUTPUT_DIR, "table1_comparison.csv")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["Variant", "CBAM", "HAFL", "Accuracy", "Macro-F1", "Sharps-F1", "Inf_ms"])
        w.writerows(rows)
    print(f"\n[CSV] Table 1 saved → {csv_path}")

def print_table2(results: dict):
    """Table 2: Per-class F1 across all variants — paste into paper appendix."""
    if not results:
        return
    # Use class list from first available result
    first = next(iter(results.values()))
    class_names = sorted(first["per_class"].keys())

    print("\n" + "=" * 110)
    print("TABLE 2 — Per-Class F1 Score Across Ablation Variants")
    print("=" * 110)
    col_w = 12
    header = f"{'Class':<38}" + "".join(f"{v:<{col_w}}" for v in VARIANTS if v in results)
    print(header)
    print("-" * 110)

    rows = []
    for cls in class_names:
        row = [cls]
        line = f"{cls:<38}"
        for v in VARIANTS:
            if v not in results:
                continue
            f1 = results[v]["per_class"].get(cls, 0.0)
            line += f"{f1:<{col_w}.4f}"
            row.append(f"{f1:.4f}")
        print(line)
        rows.append(row)

    print("=" * 110)

    import csv
    csv_path = os.path.join(OUTPUT_DIR, "table2_perclass_f1.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["Class"] + [v for v in VARIANTS if v in results])
        w.writerows(rows)
    print(f"\n[CSV] Table 2 saved → {csv_path}")
# ── Run results table ──
results = load_results()
if not results:
    print('[ERROR] No results found. Train at least one variant first.')
else:
    print_table1(results)
    print_table2(results)


[WARN] Missing results for variant A_baseline: e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\outputs\A_baseline\per_class_f1.json
[WARN] Missing results for variant B_cbam_only: e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\outputs\B_cbam_only\per_class_f1.json
[WARN] Missing results for variant C_hafl_only: e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\outputs\C_hafl_only\per_class_f1.json

TABLE 1 — BioLiteNet Ablation Study & Performance Comparison
Variant               CBAM   HAFL   Accuracy   Macro-F1   Sharps-F1   Inf. (ms) 
----------------------------------------------------------------------------------------------------
D BioLiteNet           ✓      ✓      96.97%     0.9680     0.9864     21.07 ms  

[CSV] Table 1 saved → e:\- CS\6th Semester\ML Lab\Project\BioLiteNet\outputs\table1_comparison.csv

TABLE 2 — Per-Class F1 Score Across Ablation Variants
Class                                 D_BioLiteNet
------------------------------------------------------------------------